# Roman Urdu Hate Speech Detection — Training (Kaggle GPU)

Auto-generated from `src/` by `scripts/build_kaggle_notebook.py` — **do not hand-edit this notebook**; change the source files and rerun the builder instead.

Before running: Settings → Accelerator → GPU T4 x1, and Internet → On.

In [ ]:
!pip install -q -U "transformers>=4.42" "datasets>=2.19" accelerate scikit-learn sentencepiece


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from transformers import Trainer


def tokenize_batch(examples, tokenizer, max_length: int = 128):
    return tokenizer(examples["tweet"], truncation=True, max_length=max_length)


def compute_class_weights(labels, num_labels: int) -> torch.Tensor:
    weights = compute_class_weight("balanced", classes=np.arange(num_labels), y=np.asarray(labels))
    return torch.tensor(weights, dtype=torch.float32)


class WeightedLossTrainer(Trainer):
    """A Trainer that applies a per-class weighted cross-entropy loss, so the
    5-class fine-grained task (Normal is ~8x more common than Profane) doesn't just
    collapse to predicting the majority class."""

    def __init__(self, *args, class_weights: torch.Tensor | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


def predict_logits(model, features, batch_size: int, device: str, collate_fn) -> np.ndarray:
    """Batched forward pass over tokenized features (input_ids/attention_mask
    only), returning logits as a numpy array. Shared by evaluate.py and the Kaggle
    end-to-end driver."""
    loader = DataLoader(
        features.with_format("torch", columns=["input_ids", "attention_mask"]),
        batch_size=batch_size,
        collate_fn=collate_fn,
    )

    all_logits = []
    model.eval().to(device)
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.cpu().numpy())

    return np.concatenate(all_logits, axis=0)


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)


def compute_metrics_dict(y_true, y_pred, label_names: list[str], split: str) -> dict:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(label_names)), average=None, zero_division=0
    )
    per_class = [
        {"label": name, "precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}
        for name, p, r, f, s in zip(label_names, precision, recall, f1, support)
    ]
    per_class.sort(key=lambda row: row["f1"])  # worst classes first

    return {
        "split": split,
        "n": len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "per_class": per_class,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=range(len(label_names))).tolist(),
        "label_names": label_names,
    }


def compute_metrics_for_trainer(eval_pred) -> dict:
    """Flat scalar dict for HF Trainer(compute_metrics=...) -- Trainer logging and
    metric_for_best_model need flat float values, not the nested breakdown above."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0)),
    }


In [ ]:
import argparse
import json
from pathlib import Path
from datasets import Dataset, DatasetDict, load_dataset

DATASET_NAME = "community-datasets/roman_urdu_hate_speech"

def load_ruhsold(config_name: str):
    # Some community datasets ship a legacy Python loading script that recent
    # `datasets` versions refuse to execute; fall back to the Hub's
    # auto-converted Parquet branch, which needs no remote code.
    try:
        return load_dataset(DATASET_NAME, config_name)
    except Exception:
        return load_dataset(DATASET_NAME, config_name, revision="refs/convert/parquet")


def _collect_labeled_pool(raw: DatasetDict):
    """This HF Hub port of RUHSOLD has real split-integrity issues, confirmed by
    inspection: `test` ships with every label withheld (None) for both configs --
    almost certainly reserved for a private shared-task leaderboard rather than a
    loading bug -- and `Fine_Grained`'s `validation` is an exact duplicate of its
    `train` (same 7,208 tweets, same labels, same order). `Coarse_Grained`'s
    `validation` is a genuine, distinct 800-tweet split.

    Rather than trusting the upstream split boundaries, pool every *uniquely
    labeled* tweet across `train` + `validation` (deduplicated by tweet text, which
    also neutralizes the Fine_Grained duplicate-validation bug) and re-split it
    ourselves below. `test` is dropped entirely -- it carries no usable labels.
    """
    tweets, labels = [], []
    seen = set()
    n_unlabeled, n_duplicate = 0, 0
    for split_name in ("train", "validation"):
        if split_name not in raw:
            continue
        for tweet, label in zip(raw[split_name]["tweet"], raw[split_name]["label"]):
            if label is None:
                n_unlabeled += 1
                continue
            if tweet in seen:
                n_duplicate += 1
                continue
            seen.add(tweet)
            tweets.append(tweet)
            labels.append(label)
    print(
        f"  pooled {len(tweets)} uniquely-labeled tweets from train+validation "
        f"(dropped {n_unlabeled} unlabeled, {n_duplicate} duplicate)"
    )
    return tweets, labels


def build_splits(config_name: str, val_fraction: float, test_fraction: float, seed: int) -> DatasetDict:
    raw = load_ruhsold(config_name)
    tweets, labels = _collect_labeled_pool(raw)

    class_label_feature = raw["train"].features["label"]
    pool = Dataset.from_dict({"tweet": tweets, "label": labels}).cast_column("label", class_label_feature)

    holdout_fraction = val_fraction + test_fraction
    train_holdout = pool.train_test_split(test_size=holdout_fraction, stratify_by_column="label", seed=seed)
    val_test = train_holdout["test"].train_test_split(
        test_size=test_fraction / holdout_fraction, stratify_by_column="label", seed=seed
    )

    return DatasetDict(train=train_holdout["train"], validation=val_test["train"], test=val_test["test"])


def label_names_for(splits: DatasetDict) -> list[str]:
    return splits["train"].features["label"].names


In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    set_seed,
)


# ---- config ------------------------------------------------------------------
MODEL_NAME = "xlm-roberta-base"
CONFIG_NAME = "Fine_Grained"
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1
SEED = 42
MAX_SEQ_LENGTH = 128
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 2e-5

ON_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path(__file__).resolve().parent.parent
MODEL_OUTPUT_DIR = OUTPUT_ROOT / "outputs" / f"xlmr-{CONFIG_NAME.lower()}"
RESULTS_PATH = OUTPUT_ROOT / "results" / "metrics.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  Output root: {OUTPUT_ROOT}")

# ---- data ----------------------------------------------------------------------
set_seed(SEED)
splits = build_splits(CONFIG_NAME, VAL_FRACTION, TEST_FRACTION, SEED)
label_names = label_names_for(splits)
num_labels = len(label_names)
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in enumerate(label_names)}

train_ds = splits["train"].rename_column("label", "labels")
eval_ds = splits["validation"].rename_column("label", "labels")
test_ds = splits["test"]
print(f"train={len(train_ds)}  val={len(eval_ds)}  test={len(test_ds)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
)


def tokenize(examples):
    return tokenize_batch(examples, tokenizer, MAX_SEQ_LENGTH)


train_features = train_ds.map(tokenize, batched=True, remove_columns=["tweet"])
eval_features = eval_ds.map(tokenize, batched=True, remove_columns=["tweet"])
print(f"{len(train_features)} train / {len(eval_features)} val tokenized features")

class_weights = compute_class_weights(train_ds["labels"], num_labels)
print(f"Class weights ({label_names}): {class_weights.tolist()}")

# ---- train -----------------------------------------------------------------------
steps_per_epoch = -(-len(train_features) // TRAIN_BATCH_SIZE)  # ceil div
total_steps = int(steps_per_epoch * NUM_EPOCHS)
warmup_steps = int(total_steps * 0.1)

training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    fp16=(DEVICE == "cuda"),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=eval_features,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics_for_trainer,
    class_weights=class_weights,
)
trainer.train()

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))
print(f"Saved model to {MODEL_OUTPUT_DIR}")

# ---- evaluate on held-out test split ----------------------------------------------
test_features = test_ds.map(tokenize, batched=True, remove_columns=["tweet"])
logits = predict_logits(model, test_features, EVAL_BATCH_SIZE, DEVICE, DataCollatorWithPadding(tokenizer))
preds = np.argmax(logits, axis=-1)

results = compute_metrics_dict(test_ds["label"], preds.tolist(), label_names, split="test")

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
all_results = json.loads(RESULTS_PATH.read_text(encoding="utf-8")) if RESULTS_PATH.exists() else {}
all_results[CONFIG_NAME] = results
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

print(f"Accuracy: {results['accuracy']:.4f}  |  Macro-F1: {results['macro_f1']:.4f}  |  Weighted-F1: {results['weighted_f1']:.4f}")
print(f"Saved metrics to {RESULTS_PATH}")


Outputs are written under `/kaggle/working/outputs` and `/kaggle/working/results`. Pull them back locally with:

```
kaggle kernels output <username>/<slug> -p ./kaggle_output
```